# Pre-processing News Data

## Bài toán
Dữ liệu gồm n văn bản phân vào 10 chủ đề khác nhau. Cần biễu diễn mỗi văn bản dưới dạng một vector số thể hiện cho nội dụng của văn bản đó.

## Mục lục
- Load dữ liệu từ thư mục
- Loại bỏ các stop words
- Sử dụng thư viện để mã hóa TF-IDF cho mỗi văn bản

## Phương pháp mã hóa: TF-IDF
Cho tập gồm $n$ văn bản: $D = \{d_1, d_2, ... d_n\}$. Tập từ điển tương ứng được xây dựng từ $n$ văn bản này có độ dài là $m$
- Xét văn bản $d$ có $|d|$ từ và $t$ là một từ trong $d$. Mã hóa tf-idf của $t$ trong văn bản $d$ được biểu diễn:
\begin{equation}
    \begin{split}
        \text{tf}_{t, d} &= \frac{f_t}{|d|} \\
        \text{idf}_{t, d} &= \log\frac{n}{n_t}, \ \ \ \ n_t = |\{d\in D: t\in d\}| \\
        \text{tf-idf}_{t d} &= \text{tf}_{t, d} \times \text{idf}_{t, d}
    \end{split}
\end{equation}

- Khi đó văn bản $d$ được mã hóa là một vector $m$ chiều. Các từ xuất hiện trong d sẽ được thay bằng giá trị tf-idf tương ứng. Các từ không xuất hiện trong $d$ thì thay là 0

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_files
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

%matplotlib inline

c:\Users\Lenovo\miniconda3\Lib\site-packages\pyvi\ViTokenizer.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  model = pickle.load(fin)


## Load dữ liệu từ thư mục

Cấu trúc thư mục như sau 

- data/news_vnexpress/

    - Kinh tế: 
        - bài báo 1.txt 
        - bài báo 2.txt 
    - Pháp luật
        - bài báo 3.txt 
        - bài báo 4.txt 

In [2]:
INPUT = 'news_vnexpress'
os.makedirs("images",exist_ok=True)  # thư mục lưu các các hình ảnh trong quá trình huấn luyện và đánh gía

In [3]:
# statistics
print('Các nhãn và số văn bản tương ứng trong dữ liệu')
print('----------------------------------------------')
n = 0
for label in os.listdir(INPUT):
    print(f'{label}: {len(os.listdir(os.path.join(INPUT, label)))}')
    n += len(os.listdir(os.path.join(INPUT, label)))

print('-------------------------')
print(f"Tổng số văn bản: {n}")

Các nhãn và số văn bản tương ứng trong dữ liệu
----------------------------------------------
doi-song: 120
du-lich: 54
giai-tri: 201
giao-duc: 105
khoa-hoc: 144
kinh-doanh: 262
phap-luat: 59
suc-khoe: 162
the-thao: 173
thoi-su: 59
-------------------------
Tổng số văn bản: 1339


In [4]:
# load data
data_train = load_files(container_path=INPUT, encoding="utf-8")
print('mapping:')
for i in range(10):
    print(f'{data_train.data[i][:20]} -{data_train.target_names[data_train.target[i]]}- type :{data_train.target[i]}- {data_train.filenames[i]}')
for i in range(len(data_train.target_names)):
    print(f'{data_train.target_names[i]} - {i}')

print('--------------------------')
print(data_train.filenames[0:10])
# print(data_train.data[0:1])
print(data_train.target[0:1])
print(data_train.data[0:1])
print(len(data_train.target_names))
print("\nTổng số  văn bản: {}" .format( len(data_train.filenames)))

mapping:
Mời độc giả đặt câu  -khoa-hoc- type :4- news_vnexpress\khoa-hoc\00133.txt
(Nguồn và ảnh: Trung -suc-khoe- type :7- news_vnexpress\suc-khoe\00102.txt
Lợi suất trái phiếu  -kinh-doanh- type :5- news_vnexpress\kinh-doanh\00240.txt
Leanne và Steve Ford -doi-song- type :0- news_vnexpress\doi-song\00085.txt
Chương trình ưu đãi  -kinh-doanh- type :5- news_vnexpress\kinh-doanh\00123.txt
Đang dùng bữa cùng g -suc-khoe- type :7- news_vnexpress\suc-khoe\00153.txt
Trong buổi giới thiệ -giai-tri- type :2- news_vnexpress\giai-tri\00159.txt
Zhou biết về khái ni -doi-song- type :0- news_vnexpress\doi-song\00031.txt
Theo quy định của Bộ -suc-khoe- type :7- news_vnexpress\suc-khoe\00049.txt
Theo hai báo cáo mới -suc-khoe- type :7- news_vnexpress\suc-khoe\00160.txt
doi-song - 0
du-lich - 1
giai-tri - 2
giao-duc - 3
khoa-hoc - 4
kinh-doanh - 5
phap-luat - 6
suc-khoe - 7
the-thao - 8
thoi-su - 9
--------------------------
['news_vnexpress\\khoa-hoc\\00133.txt'
 'news_vnexpress\\suc-khoe\\00102.tx

## Chuyển dữ liệu dạng text về ma trận (n x m) bằng TF-IDF

* Bạn cần viết đoạn mã tương ứng trong cell bên dưới. Theo các bước được gợi ý

In [18]:
# load dữ liệu các stopwords 
with open("vietnamese-stopwords.txt",'r',encoding="utf-8") as f:
    stop_words=[line.strip() for line in f]
stop_words=[word.replace(" ","_") for word in stop_words]
for word in stop_words[:20]:
    print(word)
#---> Code ở đây
# Code tay without function 
from collections import Counter
import math
stop_words=Counter(stop_words)
global_dict= Counter()#Một cái set để lưu toàn bộ từ trong văn bẳn 
global_token=[]# List của các token của từng văn bản 

for data in data_train.data:
    tokens=[token.strip() for token in ViTokenizer.tokenize(data).split() if token not in stop_words]

    global_dict.update(set(tokens))# Update dần dần các  token trong từ điển
    global_token.append(Counter(tokens))
for token in global_dict:# Tính IDF
    global_dict[token]=math.log10(len(data_train.data)/global_dict[token])
for text_token in global_token:#Tính tf-d
    for token in text_token:
        text_token[token]/=text_token.total()
# Chuyển hoá dữ liệu text về dạng vector TF 
#     - loại bỏ từ dừng
#     - sinh từ điển

#---> Code ở đây
# Tạo một numpy 2d để lưu 
arr=np.zeros((len(data_train.data),len(global_dict)))
vocabulary = list(global_dict.keys())
word2idx = {word: i for i, word in enumerate(vocabulary)}
for i in range(len(data_train.data)):
    for token in global_token[i]:
        arr[i][word2idx[token]]=global_token[i][token]*global_dict[token]
    
# Hàm thực hiện chuyển đổi dữ liệu text thành dữ liệu số dạng ma trận 
# Input: Dữ liệu 2 chiều dạng numpy.array, mảng nhãn id dạng numpy.array 
X=arr
Y=data_train.target

a_lô
a_ha
ai
ai_ai
ai_nấy
ai_đó
alô
amen
anh
anh_ấy
ba
ba_ba
ba_bản
ba_cùng
ba_họ
ba_ngày
ba_ngôi
ba_tăng
bao_giờ
bao_lâu


In [41]:
with open("vietnamese-stopwords.txt",'r',encoding="utf-8") as f:
    stop_words=[line.strip() for line in f]
stop_words=[word.replace(" ","_") for word in stop_words]
for word in stop_words[:20]:
    print(word)

def vietnamese_tokenizer(text):
    tokens=ViTokenizer.tokenize(text.strip())
    return tokens.split()
count_tokenizers=CountVectorizer(tokenizer=vietnamese_tokenizer,stop_words=stop_words)#Phễu lọc
vector_extracted=Pipeline(steps=[("token",count_tokenizers),("tf-idf",TfidfTransformer())])#Quy trình xử lý 

data_preprocessed =vector_extracted.fit_transform(data_train.data)
X = data_preprocessed # thuoc tinh
Y = data_train.target #nhan

a_lô
a_ha
ai
ai_ai
ai_nấy
ai_đó
alô
amen
anh
anh_ấy
ba
ba_ba
ba_bản
ba_cùng
ba_họ
ba_ngày
ba_ngôi
ba_tăng
bao_giờ
bao_lâu


c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [37]:
idx=10
print(X[idx])
print(Y[idx])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 289 stored elements and shape (1, 12797)>
  Coords	Values
  (0, 81)	0.015211595534715633
  (0, 100)	0.020769547555053954
  (0, 156)	0.023262873797037953
  (0, 188)	0.04722498744523733
  (0, 269)	0.03324036700472501
  (0, 392)	0.024783841259467605
  (0, 397)	0.034232954192352914
  (0, 418)	0.048918066523847725
  (0, 662)	0.022769929356223215
  (0, 909)	0.033182248640039956
  (0, 1194)	0.05117031195708949
  (0, 1209)	0.10234062391417897
  (0, 1219)	0.05117031195708949
  (0, 1271)	0.016005705134410072
  (0, 1590)	0.034926623903062066
  (0, 1631)	0.023493949966261335
  (0, 1783)	0.04734509560871461
  (0, 1866)	0.05117031195708949
  (0, 2076)	0.014547704347804936
  (0, 2101)	0.02693672471116765
  (0, 2111)	0.02577563465433512
  (0, 2135)	0.04322051581452055
  (0, 2140)	0.01566196368628259
  (0, 2159)	0.01608450478874651
  (0, 2170)	0.02950839772591026
  :	:
  (0, 12272)	0.021259537682082122
  (0, 12454)	0.0758997005019029
  (0, 1

In [38]:
sum(sum(X[100].toarray() != 0))

np.int64(289)

In [39]:

print(X[100].toarray())

[[0.         0.         0.         ... 0.         0.14048828 0.        ]]
